### Beyond the Squeaky Wheel: 311 Engagement & Equity Analysis
### Notebook 2: 311 Service Request Keyword Classifier Model

Classifies 311 service requests into 77 thematic categories using keyword matching. Takes a prepared CSV of service request types as input and outputs binary category matches per row, plus a per-city summary table.

In [ ]:
# Step 1: Import libraries

import pandas as pd
import re

# to clean and normalize special characters
import unicodedata

# to show status bar in model
from tqdm.auto import tqdm
tqdm.pandas()

In [ ]:
# Step 2: Upload data

SR_311 = pd.read_csv("INSERT FILE PATH: list of all 311 SRs CSV")


In [ ]:
# Sanity Check: review uploaded data

print(f"Rows: {SR_311.shape[0]}")
print(f"Columns: {SR_311.shape[1]}")
print("--------------")
print(f"Categories: {SR_311.columns.tolist()}")
print("--------------")
SR_311.sample(5)

Rows: 677913
Columns: 4
--------------
Categories: ['City', 'Department', 'Service_Request', 'Count']
--------------


,City,Department,Service_Request,Count
298603,Albuquerque,NaN,PARKS-Open Space - Other - Graffiti offf walki...,1
57726,Albuquerque,NaN,GSD - Facility Maintenance - A/C not working,12
188427,Albuquerque,NaN,CCC - Other - Chair in the road in left lane a...,1
412433,Albuquerque,NaN,CCC - Other - NMDOT dump truck losing gravel ...,1
588920,Albuquerque,NaN,Solid Waste - Trash/Recycle Cart - SWD: Trash ...,1


In [ ]:
# Step 3: Prep data
    # (remove Departments column)

consolidated = (SR_311.groupby(['City', 'Service_Request'])['Count'].sum().reset_index())

In [ ]:
# Sanity Check: review consolidated data

print(f"Rows: {consolidated.shape[0]}")
print(f"Columns: {consolidated.shape[1]}")
print("--------------")
print(f"Categories: {consolidated.columns.tolist()}")
print("--------------")
consolidated.sample(5)

Rows: 677816
Columns: 3
--------------
Categories: ['City', 'Service_Request', 'Count']
--------------


,City,Service_Request,Count
299323,Albuquerque,No Value - Weed & Litter Residential,3
519094,Albuquerque,Solid Waste - Large Item Pick Up - Friday- Gat...,1
17551,Albuquerque,"Abandoned Vehicle - Poor shape, damage on the...",1
318838,Albuquerque,Non City Request General FAQ - Donate stuffed ...,1
73237,Albuquerque,Animal Welfare Field Dispatch - Four dogs at l...,1


In [ ]:
# Step 4: Create match key
    # Concatenate city and service request first, then clean the combined string

consolidated['Combined_Match'] = consolidated['City'] + ' - ' + consolidated['Service_Request']

In [ ]:
# Sanity Check: review consolidated data

print(f"Rows: {consolidated.shape[0]}")
print(f"Columns: {consolidated.shape[1]}")
print("--------------")
print(f"Categories: {consolidated.columns.tolist()}")
print("--------------")
consolidated.sample(5)

Rows: 677816
Columns: 4
--------------
Categories: ['City', 'Service_Request', 'Count', 'Combined_Match']
--------------


,City,Service_Request,Count,Combined_Match
43672,Albuquerque,Animal Welfare - Other - Request to impound st...,1,Albuquerque - Animal Welfare - Other - Request...
7859,Albuquerque,ACS - Other - Request ACS check out a person h...,1,Albuquerque - ACS - Other - Request ACS check ...
275212,Albuquerque,Fire - Illegal Fireworks - Illegal Fire works,14,Albuquerque - Fire - Illegal Fireworks - Illeg...
130163,Albuquerque,"Animal Welfare General FAQ - Thank you, Albert...",1,Albuquerque - Animal Welfare General FAQ - Tha...
286975,Albuquerque,HHH - Health Housing Homlessness -FAQ - Mats 5...,1,Albuquerque - HHH - Health Housing Homlessness...


In [ ]:
# Step 5: Text cleaning on Combined_Match
        # Clean after combining — must be same order as match notebooks for consistent keys
        # Special character corruption is siginificant issue, test/assess SR rows for complete correction - will cause MatchKey join failures


def normalize_chars(text):
    # normalize unicode to closest ASCII equivalent
    return unicodedata.normalize('NFKD', str(text)).encode('ascii', 'ignore').decode('ascii')

#### Punctuation & spacing fixes
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.lower() 
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('&amp;', 'and', regex=False)            # converts & and invisible characters
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('&', 'and', regex=False)                # converts &
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('_', ' ', regex=False)                  # removes _
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace(',', '', regex=False)                   # removes ,
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('\t', ' ', regex=False)                 # removes tab-spaces
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('\u2019', "'", regex=False)             # converts right single quote
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('\u2018', "'", regex=False)             # converts left single quote
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('\u201c', '"', regex=False)             # converts left double quote
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('\u201d', '"', regex=False)             # converts right double quote

#### Special character/corrupted letters fixes
            # corrupted em dash
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('‚äì', '-', regex=False)
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('‚äî', '-', regex=False)
            # corrupted accented characters
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('√©', 'e', regex=False)
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('√≥', 'o', regex=False)
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('√°', 'a', regex=False)
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('√≠', 'i', regex=False)
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('√±', 'n', regex=False)
        # corrupted non-breaking space
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('¬†', ' ', regex=False)

#### Denver emails / syntax mess
        # corrupted apostrophe
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('‚äô', "'", regex=False)
        # corrupted open/close double quotes
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('‚äú', '"', regex=False)
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('‚äù', '"', regex=False)
        # corrupted open/close single quotes
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('‚äò', "'", regex=False)
        # corrupted ellipsis
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('‚ä¶', '...', regex=False)
        # trademark symbol
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('‚ñ¢', '', regex=False)
        # inverted exclamation (Spanish)
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('¬°', '', regex=False)

# Overall / final cleanup
consolidated['Combined_Match'] = consolidated['Combined_Match'].apply(normalize_chars)                              # standardizes special characters to ASCII/unicode/encoding
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace(r'\s+', ' ', regex=True).str.strip()    # removes double-spaces
consolidated['Combined_Match'] = consolidated['Combined_Match'].str.lower()                                         # converts all text lower case


# consolidated['Combined_Match'] = consolidated['Combined_Match'].str.replace('/', ' ', regex=False)                # intentionally keeping / , not running this line


In [ ]:
# Sanity Check: review prepped data

print(f"Rows: {consolidated.shape[0]}")
print(f"Columns: {consolidated.shape[1]}")
print("--------------")
print(f"Categories: {consolidated.columns.tolist()}")
print("--------------")
consolidated.sample(5)

Rows: 677816
Columns: 4
--------------
Categories: ['City', 'Service_Request', 'Count', 'Combined_Match']
--------------


,City,Service_Request,Count,Combined_Match
648066,Albuquerque,Transit General FAQ - 2024 Memorial Day,1,albuquerque - transit general faq - 2024 memor...
626472,Albuquerque,Solid Waste General FAQ - Where can I take my ...,1,albuquerque - solid waste general faq - where ...
251183,Albuquerque,DMD-Traffic Signal - Median red bulb out,1,albuquerque - dmd-traffic signal - median red ...
319731,Albuquerque,Non City Request General FAQ - Federal Gov,45,albuquerque - non city request general faq - f...
237019,Albuquerque,DMD General FAQ - #= Oversized permit,1,albuquerque - dmd general faq - #= oversized p...


In [ ]:
# Step 6: Define categories and keywords
        # Service Requests can be in multiple categories

KEYWORDS = {

# Step 6A: Core/Related Categories for Analysis

    'Bulky_Waste': [
        'bulky', 'bulk', 'bulk item', 'bulk pickup', 'bulk pick-up', 'bulk waste', 'bulk trash',
        'bulk collection', 'bulk removal', 'bulk-schedule',
        'christmas tree', 'special collection', 'fly dumping', 'fly tipping', 'mattress', 'special pickup',
        'furniture', 'appliance', 'tire pickup', 'tire pick up', 'debris pick up', 'item pickup',
        'junk accumulation', 'junk on property', 'junk/accumulation',
        #'junk',
        'illegal dump', 'illegal dumping', 'dumping', 'dumped',
        'junk removal', 'large item', 'white goods',
        'metal/household', 'electronic waste', 'e-waste',], # type: ignore
    # NOTE: Junk may be used on its own BUT it collects all "junk vehicle" SRs, review results/use as needed
    # NOTE: Bulk picks up some [bulk] emails in Denver, review results
    # NOTE: Furniture and Appliance pick up some Consumer and Transportation SRs in New York and Los Angeles, review results

    'Dead_Animal': [
        'dead animal', 'dead bird', 'dead cat', 'dead dog', 'animal - dead',
        'roadkill', 'animal carcass', 'carcass',
        'dead animal pickup', 'dead animal removal',
        'animal removal', 'animal collection'],

    'Yard_Waste': [
        'yard waste', 'yard debris', 'yardwaste', 'yard trimming',
        'brush collection', 'brush service', 'brush pickup', 'leaf and brush', 'brush pick up', 
        'leaf pickup', 'brush disposal', 'leaf pick up','leaf collection', 'leaf removal',
        'green waste', 'compost',],

    'Garbage': [
        'garbage', 'trash', 'litter', 'rubbish', 'refuse', 
        'broken glass', 'broken-glass',
        'waste collection', 'waste pickup', 'waste removal', 'waste management', 'waste service', 'bin pickup', 'bin pick up',
        'missed pickup', 'missed collection', 'missed trash', 'missed garbage', 'missed pick up',
        'solid waste', 'overflowing bin', 'overflowing container', 'overflowing dumpster', 'container overflow', 'overflowing receptacle',
        'recycling', 'recyclable', 'recycle', 
        'sanitation pickup', 'sanitation collection', 'collection truck', 'esd collections',
        'debris removal', 'and debris', 'remove debris', 'debris pickup', 'debris in', 'debris on', 'debris issue',
        'carts', 'trash cart', 'recycle cart', 'missing cart', 'cart issue', 'cart concern', 'garbage cart',
        'cart service', 'cart pickup', 'cart violation', 'cart pick up', 'cart repair', 'cart at', 'cart left', 'cart on', 'cart pick',
        'hauler', 'trash pickup', 'damaged cart', 'issued cart', 'container left',
        'dumpster', 'bin collection', 'bins', 'big belly', 'totter', 'street receptacle', 'corner can', 'park cans',
        'rollout cart', 'rollout violation', 'roll out left', 'wire basket',
        'left curbside', 'items at curb', 'roll out', 'rollout',
        'sidewalk cleaning', 'alley clean', 'dirty condition', 'way cleanup', 'dirty street',
        'medical waste',
        'animal waste', 'human waste', 'feces', 'dog waste', 
        'dirty alley', 'alley debris',],
    # NOTE: Carts picks up shopping carts SRs, review results
    # NOTE: Bin, Debris, and Cart too broad to use on their own
    # NOTE: Dirty Condition also in Vacant_Property, review results

    'Missing_Garbage': [
        'missed', 'missing',],
    # NOTE: Partner to Bulky_Waste, Yard_Waste, and Garbage categories
    # NOTE: Identifies service/collection issues and missing bins/carts rather than actual waste accumulation/litter/etc.

    'Graffiti': [
        'graffiti', 'graffitti', 'grafitti',
        'tagging', 'tagged', 'tagger',
        'illegal posting', 'illegal postings', 'illegal post', 'illegal flyer', 'illegal signs',
        'spray paint', 'spraypaint', 'oops tag',],

    'Noise': [
        'noise', 'noisy', 'loud', 'loudness',
        'sound complaint', 'sound violation',
        'noise complaint', 'noise violation', 'noise disturbance',
        'barking', 'barking dog',
        'music complaint', 'amplified sound', 'amplified noise',
        'neighbor noise', 'dumpster noise'],

    'Pothole': [
        'pothole', 'pot hole', 'pot-hole',
        'pavement defect', 'pavement failure', 'pavement damage',
        'road defect', 'carriageway defect', 'street subsidence', 'depression',
        'asphalt defect', 'asphalt repair', 'asphalt damage',
        'dip in road', 'dip in roadway', 'dip on roadway', 'bump in road', 'bump in roadway', 'dip in street', 'bump in city street',
        'cave-in', 'road cave', 'road cave-in', 'cave - in', 'cave in', 'cavein',
        'sinkhole', 'sink hole', 'sink-hole', 'chuckhole', 'chuck hole',],

    'Street_Repair': [
        'street repair', 'road repair', 'road damage', 'road resurface',
        'street resurface', 'street surface', 'road surface', 'street paving', 'street resurfacing', 'street pavement', 
        'pavement repair', 'pavement crack', 'pavement cave-in', 'pavement defect', 'pavement issue',
        'repaving', 'paving', 'repaved', 'resurfacing', 'uneven surface', 'pavement maintenance',
        'alligator crack', 'road failure', 'road maintenance', 'cracked road', 'roads repair',
        'street maintenance', 'street defect', 'repair street', 'street condition',
        'plates in street', 'street plate', 'street cut', 'metal plate', 
        'shoulders', 'shoulder damage', 'shoulder repair', 'surface repair',
        'paving request', 'alley paving', 'road to be paved', 'asphalt', 'repave', 
        'inspect public way', 'road quality', 'roadway repair', 
        'alley repair', 'alley issue', 'highway condition', 'bad condition in the right of way', 'alley repave',
        'street pavement issues', 'general street inspection',],

# Step 6B: Infrastructure Categories

    'Bridges': [
        'bridge', 'bridges', 'bridge defect', 'bridge repair', 'bridge miscellaneous', 
        'bridge maintenance', 'bridge complaint', 'bridge damage', 'bridge condition',],
    # NOTE: Bridge can be used on its own

     'Fire_Hydrant': [       
        'fire hydrant', 'hydrant',],

    'Guards_Barriers': [
        'guardrail', 'guard rail', 'barricade', 'barricad', 'bollard', 'retaining wall', 'warning rail',],

    'Obstructions': [
        'obstruction', 'obstruct',
        'blocked street', 'blocked sidewalk', 'sidewalk block', 'street block', 'blocking the right of way', 
        'blocking roadway', 'blocking traffic', 'road blockage', 'street blockage',],

    'Plumbing': [
        'plumbing', 'plumber',],

     'Road_Markings': [       
        'speed hump', 'speed bump', 'speed cushion',
        'marking installation', 'line marking', 'markers/deliniators',
        'road marking', 'lane marking', 'traffic marking', 'pavement marking',
        'crosswalk', 'crosswalk marking', 
        'paint curb', 'paint legend', 'paint striping',
        'roadway marking', 'road striping', 'road stripe', 'row maintenance - paint', 'streets - paint',
        'street marking',],

    'Sidewalk': [
        'sidewalk', 
        'curb repair', 'curb damage', 'damaged curb', 'curb issues', 'curb piece','curb markings',
        'curb ramp', 'curbramp', 'curb-ramp', 'ramp defect', 'pedestrian ramp', 'curb condition',
        'curbing and berm', 'driveway berm', 'gutter repair',
        'pedestrian', 'path of travel', 'footways',
        'trip hazard', 'tripping hazard',
        'sidewalk repair', 'sidewalk damage', 'sidewalk defect', 'sidewalk bollard',],

    'Street_Lighting': [
        'street light', 'streetlight', 'street lamp', 'streetlamp',
        'light outage', 'light out', 'light repair', 'light offline',
        'navigation light', 'city light', 'alley light', 'stoplight',
        'light(s) out', 'stlight', 'light damage', 'hanging light',
        'pole down', 'light pole', 'lightpole',],

    'Street_Sweeping': [
        'street cleaning', 'street sweep', 'sweeping',
        'mechanical sweep','special sweep', 'SW-Cleaning',],

    'Traffic_Signs': [
        'damaged sign', 'missing sign', 'sign repair', 'sign replacement',
        'request new sign', 'sign new', 'signage', 'sign down',
        'traffic signage', 'wayfinding', 'sign damage',
        'traffic sign issues', 'roadway sign', 'row maintenance - sign', 'row maintenance - traffic',
        'traffic sign ', 'traffic signs', 'stop sign', 'highway sign', 'yield sign',
        'street sign', 'streetsign', 'sign street', 'street name sign', 'name street sign',
        'signage problem', 'sign replace', 'signage request',],     
    # NOTE: Lots of notes to review but generally works, give one last look

    'Traffic_Signals': [
        'traffic light', 'traffic signal', 'signal outage', 'traffic signal issues',
        'signal repair', 'signal timing', 'signal malfunction', 'signal change', 'signal traffic',
        'traffic light issue', 'traffic light timing', 'traf signal',
        'request new signal', 'red light',],

    'Sewer': [
        'sewer', 'sewage', 'sewage overflow', 'sewer backup',
        'wastewater', 'waste water',
        'sewer blockage', 'sewer overflow', 'sewer repair',],

    'Water': [
        'water main', 'water leak', 'water outage', 'water service', 'leaking water',
        'water pressure', 'water quality', 'water line', 'water system', 'water discolor', 'odorous water',
        'discharge of water', 'water waste', 'water wasting', 'water in basement', 'water related',
        'water main break', 'broken water main', 'wtr-coming-up', 'water on street',
        'stagnant water', 'pooling water', 'ponding water', 'water ponding', 
        'hot water', 'no water', 'boiler', 'water turn', 'low pressure', 'backflow', 'boil water',
        'water meter', 'water conservation', 'water valve', 'water test', 'water works',
        'frozen water', 'waste of water', 'tap water', 'tapwater', 'drinking water',],

    'Water_and_Sewer': [
        'drain repair', 'storm drain', 'drainage', 'blocked drain', 'drainage ditch',
        'flooding', 'flood', 'standing water',
        'manhole', 'lid cover', 
        'inlet cleaning', 'inlet', 'culvert',
        'catchbasin', 'catch basin', 'catch-basin', 'rain garden', 'blocked basin',
        'stormwater', 'storm water', 'backwater',
        'broken meter',
        'burst pipe', 'broken pipe', 'leaking pipe', 'main break'],
    # NOTE: Broken Meter also relevant to Utilities_Power, review results

    'Utilities_Power': [
        'utility', 'utilities', 'utility repair', 'utility complaint', 'utility cut',
        'power outage', 'electric outage', 'electricity outage',
        'electrical', 'mechanical', 
        'power line', 'downed line', 'downed wire', 'wires down',
        'air conditioning', 'no heat', 'hvac', 'no electricity', 'no power', 'insufficient heat',
        'wire and pole', 'utility pole', 'electric pole', 
        'internet/cable', 'wireless', 'cell service', 'cable', 'conduit', 'telecommunication',
        'gas leak', 'gas line', 'gas service',],
    # NOTE: Broken Meter also relevant to Water_and_Sewer, review results

# Step 6C: Transportation Categories

    'Abandoned_Vehicle': [
        'abandoned vehicle', 'abandoned car', 'abandon car', 'abandon vehicle',
        'abandoned property / vehicle', 
        'oned vehicle', 'vehicle abandon',
        'abandoned auto', 'abandoned truck', 'derelict vehicle',
        'vehicle abatement', 'junk vehicle', 'junk car',
        'inoperable vehicle', 'inoperable car',
        'vehicle removal', 'car removal',
        'junked vehicle', 'junked car'],
    # NOTE: Notable pattern of word "abandoned" typos in SRs, review/revise as needed

    'Airport': [
        'airport', 'airline', 'flight complaint',
        'noise from airport', 'aircraft noise', 'plane noise',
        'flight path', 'runway', 'faa',
        'aviation', 'aircraft'],

    'Bike_Scooter': [
        'bike', 'bicycle', 'bicycling', 'biking', 'bicyclist',
        'scooter', 'shared scooter', 
        'shared mobility', 'shared micromobility',
        'bike share', 'dockless', 'docking station', 'bike share station', 'bikeshare station',
        'bike lane', 'bike path', 'bicycle facilities', 'bike trail', 'bikeway',
        'bicycle rack', 'bike rack',],

    'Parking': [
        'parking', 'illegal parking', 'illegally park', 'parking violation', 'parking complaint', 'improperly park',
        'parking enforcement', 'double park', 'parking lane', 'parking space', 'street parking',
        'blocking fire hydrant', 'blocking intersection','blocking driveway', 
        'no parking zone', 'handicap parking', 'accessible parking',
        'parking meter', 'parking machine', 
        'alley parking', 'vehicle parking', 'oversized vehicle',
        'improperly parked', 'parking permit', 'parking per',
        'parking lot', 'parking garage',
        'blocking city right of way', 'blocked right of way', 'blocking right of way',
        'blocked drive', 'from the curb', 'inches from curb',
        'hrs marked', 'hour marked', '2 hour', '72 hour', '72-hour', 'hr parking',
        'vehicle parked', 'cars parked on lawn','car parked on lawn', 'car parked',
        'loading zone', 'time marked', 'hr marked',],
    # NOTE: Notable overlap with Vacant_Property category

    'Public_Transit': [
        'bus stop', 'bus route', 'bus complaint', 'busstop', 'bus pad', 
        'subway', 'light rail', 'streetcar', 'street car', 'bus/rail',
        'transit shelter',  'bus shelter', 
        'ferry', 'ferries',   
        'transportation',        # review results, may cut        
        'public transit', 'public transporation', 'transit complaint',],
    # NOTE: Metro too broad to use on its own

    'Taxi_Vehicles': [
        'taxi', 'taxi cab', 'taxicab', 'uber', 'limousine', 
        'vehicle for hire', 'vehicles for hire','for hire vehicle',],

    'Traffic': [
        'traffic study', 'flow of traffic', 'traffic control', 'speed control',
        'vision zero', 'speed limit', 'traffic - general issue',
        'traffic complaint', 'traffic safety', 'traffic calming', 'gridlock',
        'traffic engineering', 'traffic management',],

    'Towing': [
        'towing', 'tow lot', 'tow truck', 'waiver tow',],

# Step 6D: Parks Categories

    'Parks': [
        'park maintenance', 'park complaint', 'park damage', 'park issue', 'park improvement',
        'parks maintenance', 'parks complaint', 'parks damage', 'parks issue', 'parks improvement', 
        'ground maintenance',
        'park lighting', 'park graffiti', 'park light',
        'POPOS', 'privately owned public open space',
        'public park', 'city park', 'private park', 'parks ', 'parkland', 'county park', 
        'park rules', 'park cleanliness', 'park amenity',],
    # NOTE: Trailing space on Parks intentional, term otherwise too broad

    'Park_Features': [
        'playground', 'playground equipment',
        'recreation', 'rec center', 'recreation center',
        'sports field', 'ball park', 'ball field', 'ballfield',
        'athletic field', 'picnic', 'basketball', 'pickleball',
        'baseball field', 'soccer field', 'tennis court', 
        'swimming pool', 'swim pool', 'lifeguard', 'aquatic facility', 'aquatic facilities',],

# Step 6E: Vegetation Categories

    'Trees': [
        'tree trimming', 'tree maintenance', 'tree concerns', 'tree service', 'tree trim',
        'tree inspection', 'tree plant', 'tree prune', 'tree pruning', 'tree permit', 'tree debris', 'prune tree',
        'private tree', 'city tree', 'forestry', 'public tree', 
        'tree removal', 'tree fell', 'tree in road', 'tree damage',
        'tree safety', 'tree issue', 'protected tree', 'tree emergency', 'tree emergencies',
        'fallen tree', 'dead tree', 'damaged tree', 'remove a tree', 'tree dead', 
        'tree down', 'down tree', 'downed tree',
        'fallen limb', 'fallen branch', 'dead limb', 'tree limb', 'limb down', 'low limb', 'limbs', 'broken branch', 
        'hanging branch', 'hanging limb', 'remove stump', 'uprooted stump',
        'street tree', 'streettree', 'trees ', ' trees', 'new tree',
        'tree ', 'tree hazard',
        'treestump', 'tree stump', 'stump removal', 
        'tree obstruction', 'tree blocking'],
    # NOTE: Trailing space on Tree intentional, term otherwise too broad

    'Vegetation': [
        'overgrowth', 'overgrown', 'overgrown grass', 'grass over', 'over grown',
        'weeds', 'tall grass', 'high grass', 'tall weed', 'high weed', 'weed removal', 'weed cleanup',
        'weeds/grass', 'grass/weed', 'grass and weed', 'weed and grass', 'weeds and grass',
        'grass cut', 'cut grass', 
        'planter', 'planting',
        'island maintenance', 'median maintenance', 'right of way - maintenance',
        'bushes', 'bush', 'vegetation', 'shrub', 'shrub prune', 'shrub pruning',
        'landscape', 'landscaping', 
        'leaf blowing', 'leaf blow', 'blowing leaf', 'blowing leaves', 'illegal blowing',
        'mowing', 'right of way mow',],
    # NOTE: Yard_Waste category has keywords for brush trimmings, yard waste, brush collection, etc.

# Step 6F: Property and Business Categories

    'Alcohol_Liquor': [
        'liquor', 'liquor license', 'liquor store',
        'alcohol', 'alcohol complaint', 'alcohol permit', 'alcohol violation', 'alcohol license',
        'wine', 'wine and', 'beer permit',],

    'Commercial': [
        'storefront',
        'vending', 'vendor', 'sidewalk vendor',
        'mixeduse', 'mixed use', 'mixed-use', 'commercial', 'small business', 'industrial',
        'business', 'retail',
        'non-residential',
        'nightclub',
        'food truck', 'mobile food', 'outdoor dining', 
        'employee', 'workplace', 'employment', 
        'warehouse', 'store front', 'business location', 'commercial property', 'commercial building',],
    # NOTE: Approximate/bundled category, review/revise per needs

    'Consumer': [
        'consumer', 'consumer complaint',],

    'Residential': [
        'residential', 'residence', 'resident', 'residency',
        'short term rental', 'home occupation', 'short term rent', 'airbnb', 'vrbo', 'vacation rental', 'stro violation',
        'single family', 'single-family',
        'apartment', 'duplex', 'triplex', 
        'tenant', 'landlord', 'renter', 'rental property', 'eviction', 'rental registration', 'rental licens',
        'housing', 'house', 'boarding home', 
        'multi family', 'multi-family', 'multifamily',
        'condo', 'condominium',
        'townhouse', 'townhome',
        'property owner', 'homeowner', 'household',
        'section 8', 'housing voucher', 'affordable housing', 'group home',
        'home repair', 'illegal occupancy', 
        'living conditions', 'living condition', 'overcrowding', 'over crowding',
        'accessory dwelling unit', 'accessory dwelling', 'cottage',
        'mobile home', 'manufactured home',],
    # NOTE: ADU too broad to use on its own

    'Tobacco_Smoking': [
        'tobacco', 'smoking', 'vaping', 'vape',
        'cigaratte', 'cannabis', 'marijuana', 
        'smokeshop', 'smoke shop',],

    'Vacant_Property': [
        'vacant property', 'vacant lot', 'vacant structure', 'vacant building',
        'structure vacant', 'vacant house', 'building vacant', 'vacant home',
        'dilapidated structure', 'unsafe structure', 'open and vacant', 'unsafe property',
        'abandoned building', 'abandoned structure', 'abandoned property',
        'property maintenance', 'lot condition',        # review results, may cut  
        'boarded up', 'home boarded', 'unsecured structure',
        'neglect of property', 'neglected property', 'property neglect', 'neglect of premise',],
    # NOTE: 'Damaged Property' and 'Dirty Condition' too broad to use on their own

# Step 6G: Planning and Development Categories

    'Code_Enforcement': [
        'building violation', 'building complaint', 'building collapse', 
        'code violation', 'code enforcement', 'code complaint', 'land use enforcement',
        'property violation', 'dangerous building', 'dangerous structure', 'dangerous conditions',
        'notice of violation', 'notice of enforcement', 'violation notice', 'enforcement notice',
        'dilapidated', 'unsafe structure', 'structure unsound', 'unsound structure', 'unsafe building', 'unsafe condition',
        'housing violation', 'housing complaint', 'code concern',
        'request code officer', 'building inspection', 'building inspector',
        'building maintenance',        # review results, may cut 
        'without permit', 'unpermitted', 'no permit', 'without a permit', 'w/o permit', 'w/out permit',
        'unpermitted construction', 'plumbing violation','outside storage',
        'fence complaint', 'fence violation', 'fencing', 'fence repair',
        'illegal construction', 'illegal structure'],
    # NOTE: Notable overlap with Vacant_Property category
    # NOTE: Fence too broad to use on its own?

    'Construction': [
        'construction', 'construction complaint', 'construction noise',
        'construction site', 'construction permit',
        'construction debris', 'construction zone',
        'unpermitted work', 'scaffolding'],

    'Demolition': [
        'demolition', 'demolish', 'condemn', 'collapse', 'collapsing',],

    'Illegal Use': [
        'land use enforcement', 'land use violation',
        'illegal land use', 'unpermitted land use', 'illegal use', 'unpermitted use',
        'illegal operation', 'unpermitted operation',
        'zoning violation', 'zoning enforcement',],
    # NOTE: Notable overlap with Planning_Zoning
    # NOTE: Category identifies a thematic subset of SRs

    'Planning_Zoning': [
        'zoning', 'zoning appeal', 'zoning request', 'variance',
        'planning appeal', 'planning request', 'planning', 'city planning', 
        'building permit', 'development review', 
        'certificate of demolition', 'landmark preservation',
        'design review', 'design advisory',
        'land use', 'rezoning', 'overlay', 'zon general', 
        'historic preservation', 'historic district', 'conservation district', 'vintage home', 'landmark','historic/conservation',
        'notice of intent', 'setbacks',
        'environmental review', 'impact assessment',],

# Step 6H: Public Safety Categories

    'Animal_Control': [
        #'animal', 
        'animals',
        'loose dog', 'stray dog', 'stray cat', 'tethered dog', 'livestock',
        'animal complaint', 'animal control', 'animal bite', 'animal care', 'animal welfare', 'animal service',
        'loose animal', 'animal loose', 'animal stray', 'animal running', 'animal issue',
        'animal at large', 'animal in a park', 'animal in park', 'animal-abuse',
        'animal - abuse', 'animal fight', 'animal - stray', 'animal problem', 'animal lack of care',
        'aggressive animal', 'animal failure', 'animal roadside', 'sick animal', 'animal aggressive',
        'animal cruelty', 'injured animal','animal confine', 'animal in trap', 'animal trap',
        'animal protection', 'animal attack', 'animal abuse', 'animal nuisance', 'animal owner', 
        'animal illegal', 'illegal animal', 'unsanitary animal', 'animal in vehicle',
        'animal noise', 'animal found', 'animal lost', 'animal search', 'animal - proper care',
        'rabid animal', 'domestic animal', 'stray animal', 'wild animal', 'animal - wild', 
        'abandoned animal', 'animal - abandoned', 'grazing animal',
        'unleashed', 'leash law', 'pit bull', 'pet account', 'found a pet', 'lost a pet',
        'pet sale', 'pet for sale', 'selling pet', 'pet wellness', 'pet resource', 'found pet', 'pet store',
        'vicious dog', 'dog attack', 'nuisance dog', 'dog bite', 'dangerous dog', 'dog trap', 'dogs at large',
        'wildlife', 'coyote', 'raccoon',],
    # NOTE: Animal may be used on its own BUT it collects all "Dead Animal" SRs, review results/use as needed

    'Fire_Safety': [
        'fire hazard', 'fire safety', 'fire life safety', 'fire escape',
        'fire code', 'fire station', 'fire alarm', 'smoke detector', 'smoke alarm',
        'carbon monoxide', 'explosive',],

    'Public_Safety': [
        'police', 'policing', 'non-emergency police', 'extra patrol',
        'public safety', 'narcotics',        
        'prostitution', 'loitering',
        '911', 'emergency',
        'crime', 'criminal', 'suspicious',],

    'Narcotics': [
        'drug', 'drugs', 'narcotic', 'drug dealing', 'drug activity',
        'drug paraphernalia', 'needle', 'syringe', 'injection site',
        'medical waste',        # review results, may cut  
        'substance abuse', 'overdose', 'narcan', 'illegal substance', 'sharps',],
    # NOTE: Drug should be fine to use on its own, review results

# Step 6I: Health and Environment Categories

    'Air_Pollution': [
        'air pollution', 'air quality', 'air complaint',
        'exhaust', 'emissions', 'smog',
        'smoke complaint', 'burning complaint',
        'idling vehicle', 'vehicle exhaust', 'vehicle idling',
        'particulate', 'air monitoring'],    
 
    'COVID': [
        'covid', 'covid-19', 'pandemic', 'vaccine mandate',
        'social distance', 'social distancing',
        'corona virus', 'coronavirus'],

    'Food_Safety': [
        'food borne', 'food operation', 'food complaint', 'food-borne',
        'food alert', 'food facility', 'food establishment',
        'food safety', 'food control', 'food protection', 'food poison',
        'restaurant complaint', 'health inspection',
        'unsanitary food'],
    # NOTE: Approximate/bundled category, review/revise per needs  

    'Hazardous_Conditions': [
        'mold', 'mildew', 'asbestos', 'lead', 'lead paint', 'lead concern',
        'hazardous', 'toxic', 'contamination', 'hazard', 'biohazard',
        'carbon monoxide', 'chemical', 'chemical spill', 'radioactive',
        'health hazard', 'unsafe condition', 'toxic drop site', 'tox lot',
        'open burn', 'smoke pollution', 'dust'],

    'Health': [
        'health', 'public health',],
    # NOTE: Broad category, identifying thematic syntax

    'Nuisance': [
        'nuisance',],
    # NOTE: Broad category, identifying thematic syntax

    'Odor_Smell': [
        'odor', 'smell', 'stench', 'foul smell', 'foul odor', 'odorous',
        'sewage smell', 'gas smell', 'chemical smell'],

    'Pests_Rodents': [
        'rodent', 'rat ', 'rats', 'mice', 'mouse',
        'pest', 'roaches', 'cockroach', 'bed bug', 'bedbug', 'bed-bug',
        'mosquito', 'lanternfly', 'fleas',
        'bee hive', 'beehive', 'bees or beehive', 'hornet', 'bee/wasp', 'wasp/bee', 'bee and wasp',
        'insect', 'infest', 'infestation', 'extermination'],
    # NOTE: Trailing space on Rat intentional, term otherwise too broad  

    'Sanitation': [
        'sanitation', 'sanitary',],
    # NOTE: Broad category, identifying thematic syntax

    'Snow_Ice': [
        'snow', 'ice removal', 'snow removal', 'snow and ice',
        'icy road', 'icy sidewalk', 'ice condition',
        'shoveling', 'sanding', 'slippery street',  
        'snow plowing', 'snowplow', 'snow plow',
        'salt request', 'salting', 'saltbox', 'de-icing'],

    'Storm_Weather': [
        'sandbag', 'storm debris', 'storm damage', 'storm relief',
        'hurricane', 'tornado', 'blizzard', 'flash flood', 'flash-flood', 'windstorm',
        'severe weather', 'inclement weather', 'weather shelter',
        'winter weather', 'cold weather', 'freezing weather',
        'weather siren', 'weather alert', 'hot weather', 'extreme weather',
        'weather emergency', 'disaster recovery', 'weather disaster', 'disaster prep',],
    # NOTE: Weather and Storm too broad to use on their own

# Step 6J: Social services categories

    'Adult_Senior_Services': [
        'senior', 'elder', 'elderly', 'adult service', 'veteran', 
        'social service', 'senior service', 'aging service',
        'wellbeing', 'well-being', 'wellness', 'welfare check',
        'mental health', 'mental illness',
        'assisted living',],
    # NOTE: Adult too broad too use on its own
    # NOTE: Wellbeing terms may overlap with Animal_Control, review results

    'Childrens_Services': [
        'children', 'child', 'daycare', 'day care',
        'parental', 'parents',
        'kids', 'youth', 'juvenile', 'underage',
        'childcare', 'child care',],

    'Disability_ADA': [
        'ada', 'ada access',
        'disability', 'disabled', 'disable',
        'ada ramp', 'ada issue', 
        'wheelchair', 'wheel chair',
        'accessibility', 'accessible',
        'handicap'],

    'Homeless': [
        'homeless', 'encampment', 'unhoused',
        'vagrant', 'unsheltered', 'squatter', 'panhandling',
        'homeless camp', 'tent city', 'shelter site', 'tents/sleeping',
        'illegal camping', 'illegal camp',],

    'School': [
        'school', 'afterschool', 'after school', 'education',
        'school zone', 'school crossing',],

# Step 6K: Civic categories

    'Court_Legal': [
        'court date','court', 'pretrial', 'trial request', 
        'jury duty', 'judicial', 'litigation', 'subpeona', 
        'arrested', 'arrest', 'warrant', 'probation',
        'district attorney', 'city attorney', 'lawyer', 'public defender', 'attorney', 'law firm',
        'legal claim', 'lawsuit', 'law center', 'citation appeal'],
    # NOTE: Trial too broad to use on its own
    # NOTE: Court picks up a few tennis/pickleball SRs, review results

    'Events': [
        'festival', 'convention', 'special event', 'faire',
        'event', 'conference', 'parade',
        'film permit', 'film notice', 'film location',],
    # NOTE: Event picks up 'Prevention' (in 25 service types), review results 

    'Financial': [
        'billing', 'invoice', 'payment', 'fines', 'taxes',
        'tax assessment', 'tax bill', 'tax inquiry', 'property tax', 
        'tax receipt', 'tax relief', 'tax document', 'tax credit', 'tax sale', 
        'business tax', 'tax certificate', 
        'penalty', 'revenue', 'rebate', 'deposit',
        'finance', 'financial', 'treasurer', 'treasury',               
        'bills', 'water bill', 'electric bill', 'utility bill', 'sewer bill',
        'bill dispute', 'bill copy', 'bill inquiry', 'bill adjustment',
        'refund', 'overpayment', 'lien', 'delinquent',
        'bankrupt', 'bankruptcy'],
    # NOTE: Fee, Bill, Tax, and Fine are too broad to use on their own

    'Paperwork_Records': [
        'funeral', 'birth certificate', 'marriage certificate', 'marriage license', 'divorce', 'death certificate', 
        'permit application', 'license application', 'application license', 'license renewal', 
        'business license', 'business licensing',
        'account holder', 'account information',
        'license and permit', 'permit and license', 'permits and license', 'licenses and permit',
        'citation',
        'public records', 'records request', 'open records', 'vital records'],

    'Public_Bodies': [
        'city council', 'city clerk', 'city hall', 'mayor',
        'board of supervisors', 'planning commission', 'district council', 'historical commission',
        'commission', 'assessor', 'controller', 'county clerk', 
        'department of', 'office of', 'agency',
        'public notice',
        'commission meeting', 'council meeting',
        'treasury'],
    # NOTE: Broader SR syntax/keyword usage in some cities, review results

    'Volunteering': [
        'volunteer', 'volunteering', 
        'non-profit', 'nonprofit',
        'charity', 'community service',],

    'Voting_Elections': [
        'vote', 'ballot', 'voting', 
        'election', 'campaign',
        'polling', 'voter registration',],
        
# Step 6L: Administration and Records Categories

    'Compliment': [
        'compliment', 'recognition', 'commendation',],

    'Informational': [
        'information request', 'inquiry', 'general inquiry', 'general question',
        #'information',
        'information only', 'information-only', 'directory', 'directory assistance', 'info only',
        'information call', 'information provided', 'request for information', 'information case',
        'general request', 'general information'],
    # NOTE: Category identifies queries and questions
    # NOTE: Records still usable, primarily appears to be formatting and record-keeping syntax      
    # NOTE: Information may be used on its own BUT it collects wider range of SRs, review results/use as needed

    'Testers': [
        'ignore', 'please ignore', 'test record', 'disregard',
        'do not use', 'test issue', 'test crm', 'test sr', 'testsr',],
    # NOTE: Category identifies test requests and unusable sample records. Do not include in analysis
    # NOTE: Test and Tests are too broad to use on their own
    # NOTE: "Staff Only", "Field Use", and comparable terms not included because they are real SRs generated by employees instead of residents
    }

In [ ]:
# Step 7: Create classifier function

def classify(text):
    """Returns list of matching category names. Empty list = no match."""
    text_lower = str(text).lower()
    matches = []
    for category, patterns in KEYWORDS.items():
        for pattern in patterns:
            if re.search(re.escape(pattern.lower()), text_lower):
                matches.append(category)
                break           # one match per category is enough
    return matches

In [ ]:
# Step 8: Run classifier
    # Slow step, about 40 minutes
    # Be sure to run on the prepped Combined_Match column

# Apply keyword classifier
classified_results = consolidated.copy()

classified_results['Categories'] = classified_results['Combined_Match'].progress_apply(classify)

# Binary columns per category (1 = matched, 0 = not matched)
for category in KEYWORDS.keys():
    classified_results[category] = classified_results['Categories'].apply(
        lambda cats: 1 if category in cats else 0)

# Two summary columns
category_cols = list(KEYWORDS.keys())
classified_results['Any_Match'] = (classified_results[category_cols].sum(axis=1) > 0).astype(int)
classified_results['Total_Categories'] = classified_results[category_cols].sum(axis=1)

  0%|          | 0/677816 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Sanity Check: review classifier results

print(f"Rows: {classified_results.shape[0]}")
print(f"Columns: {classified_results.shape[1]}")
print("--------------")
print(f"Categories: {classified_results.columns.tolist()}")
print("--------------")
classified_results.sample(5)

In [ ]:
# Step 9: Review results

print(f"\n=== CLASSIFICATION SUMMARY ===")
print(f"Total working rows:     {len(classified_results):,}")
print(f"Matched (any category): {classified_results['Any_Match'].sum():,}")
print(f"Unmatched:              {(classified_results['Any_Match'] == 0).sum():,}")
print(f"\nMatches per category:")
for category in KEYWORDS.keys():
    n = classified_results[category].sum()
    print(f"  {category:<20} {n:>6,}")

In [ ]:
# Step 10: Export results

classified_results.to_csv("INSERT FILE PATH: master key of all categorized 311 SRs CSV", index=False)


In [ ]:
# Step 11: Make summary table by city

category_cols = list(KEYWORDS.keys())

# total unique service request types per city
total_per_city = classified_results.groupby('City')['Combined_Match'].count().rename('Total_Request_Types')

# sum of matches per category per city
category_per_city = classified_results.groupby('City')[category_cols].sum()

# combine
city_summary = pd.concat([total_per_city, category_per_city], axis=1)

# add a total matched row and column
city_summary['Total_Matched'] = classified_results.groupby('City')['Any_Match'].sum()

# grand total row at the bottom
city_summary.loc['TOTAL'] = city_summary.sum()

# export to csv
city_summary.to_csv('city_summary_categorized_results.csv')

print(city_summary.to_string())